# Compare saved evaluation runs

Every `language-id eval` run is saved under `results/<model>_<dataset>_<timestamp>/` with its metrics, per-language scores, and plots. This notebook loads several of those **already-computed** runs back from disk and compares them side by side for post-evaluation analysis.

For example, this can be useful to answer: *given the same dataset / experiment setup, which model was best overall, and where does each one win or lose per language?*

Only runs on the **same dataset with the same per-language support** to provide a fair comparison.

## Setup

In [ ]:
import matplotlib.pyplot as plt

%matplotlib inline

from language_id.compare_experiments import (
    RESULTS_ROOT,
    discover_runs,
    load_runs,
    check_comparable,
    overview_table,
    per_language_pivot,
    style_overview,
    style_per_language,
    plot_overview,
    plot_per_language_heatmap,
    plot_disagreement,
)

# Comparison figures from this notebook are written here.
FIG_DIR = RESULTS_ROOT / 'comparisons'
FIG_DIR.mkdir(parents=True, exist_ok=True)

## Pick the runs to compare

`discover_runs()` lists every saved run id (newest first). Copy the ones you want into `RUN_IDS` below. They should share a dataset and experiment setup. Set `RUN_IDS = None` to load **all** saved runs instead.

In [2]:
for run_id in discover_runs():
    print(run_id)

nllb-lid_commonlid_20260601T172306Z
langdetect_commonlid_20260601T174442Z
glotlid_commonlid_20260601T172923Z


In [3]:
RUN_IDS = [
    'glotlid_commonlid_20260601T172923Z',
    'langdetect_commonlid_20260601T174442Z',
    'nllb-lid_commonlid_20260601T172306Z',
]
# RUN_IDS = None  # uncomment to compare every saved run

runs = load_runs(RUN_IDS)
print(f'Loaded {len(runs)} runs:', ', '.join(r.label for r in runs))

# Sanity-check that these runs are directly comparable.
warnings = check_comparable(runs)
if warnings:
    print('\n⚠️  comparability warnings:')
    for w in warnings:
        print('  -', w)
else:
    print('✓ same dataset and per-language support — directly comparable')

Loaded 3 runs: glotlid, nllb-lid, langdetect
✓ same dataset and per-language support — directly comparable


## Overview: which model won?

One row per model, sorted by macro-F1 (best first). Greener is better.

In [4]:
table = overview_table(runs)
best = table.iloc[0]
print(f"Best overall: {best['model']}  (macro-F1 {best['macro_f1']:.3f}, accuracy {best['accuracy']:.3f})")
style_overview(table)

Best overall: glotlid  (macro-F1 0.614, accuracy 0.655)


,model,dataset,n,n_languages,accuracy,macro_f1,run_id
0,glotlid,commonlid,373230,108,0.655,0.614,glotlid_commonlid_20260601T172923Z
1,nllb-lid,commonlid,373230,108,0.591,0.494,nllb-lid_commonlid_20260601T172306Z
2,langdetect,commonlid,373230,108,0.527,0.247,langdetect_commonlid_20260601T174442Z


In [ ]:
ax = plot_overview(runs)
plt.tight_layout()
fig_path = FIG_DIR / 'overview.png'
ax.figure.savefig(fig_path, dpi=150, bbox_inches='tight')
print('saved', fig_path)
plt.show()

## Per-language comparison

Per-language scores for every model, side by side. Languages are sorted by `support` (most-represented first). Switch `METRIC` to `'recall'` or `'precision'` to compare on a different axis.

In [6]:
METRIC = 'f1'  # 'f1', 'recall', or 'precision'
pivot = per_language_pivot(runs, METRIC)
style_per_language(pivot)

,model,support,glotlid,langdetect,nllb-lid
name,lang,,,,
Uzbek,uzb,43189,0.00,0.00,0.00
Indonesian,ind,33828,0.88,0.56,0.89
Malay,msa,28224,0.00,0.00,0.00
English,eng,27461,0.90,0.85,0.88
Standard Arabic,arb,26152,0.75,0.00,0.74
Vietnamese,vie,21803,0.99,0.95,0.96
Persian,fas,19318,0.99,0.99,0.01
Hausa,hau,16455,0.97,0.00,0.95
Arabic,ara,16306,0.00,0.54,0.00


### Per-language heatmap

In [ ]:
ax = plot_per_language_heatmap(runs, METRIC)
plt.tight_layout()
fig_path = FIG_DIR / f'per_language_{METRIC}_heatmap.png'
ax.figure.savefig(fig_path, dpi=150, bbox_inches='tight')
print('saved', fig_path)
plt.show()

### Where models disagree most

Languages with the largest best-minus-worst spread across models i.e. where the choice of model matters most.

In [ ]:
ax = plot_disagreement(runs, METRIC, top=15)
plt.tight_layout()
fig_path = FIG_DIR / f'disagreement_{METRIC}.png'
ax.figure.savefig(fig_path, dpi=150, bbox_inches='tight')
print('saved', fig_path)
plt.show()